In [1]:

# Instalar a biblioteca de otimização matemática
!pip -q install pulp

# Importar a biblioteca
import pulp

print("PuLP instalada com sucesso!")


PuLP instalada com sucesso!


In [3]:

# Instalar o solucionador CBC
!pip -q install "pulp[cbc]"

import pulp

# PROBLEMA 1 — MAXIMIZAÇÃO DO LUCRO
modelo = pulp.LpProblem("Maximizar_Lucro", pulp.LpMaximize)

# Variáveis de decisão
camisetas = modelo.add_variable(
    "Camisetas", lowBound=0, cat="Integer"
)
calcas = modelo.add_variable(
    "Calcas", lowBound=0, cat="Integer"
)

# Função objetivo
modelo += 34.90 * camisetas + 79.90 * calcas

# Restrições
modelo += 55 * camisetas + 100 * calcas <= 2500
modelo += 15 * camisetas + 35 * calcas <= 480
modelo += camisetas <= 30
modelo += calcas <= 15

# Resolver
resultado = modelo.solve(pulp.COIN_CMD(msg=False))

print("Status:", resultado.status_str)

if resultado.has_solution:
    print("Camisetas:", int(camisetas.varValue))
    print("Calças:", int(calcas.varValue))
    print(f"Lucro máximo: R$ {resultado.objective:.2f}")


Status: Optimal
Camisetas: 25
Calças: 3
Lucro máximo: R$ 1112.20


In [4]:

# CENÁRIO WHAT-IF — REDUÇÃO DO TEMPO

crise = pulp.LpProblem("Cenario_Crise", pulp.LpMaximize)

cam_crise = crise.add_variable(
    "Camisetas_Crise", lowBound=0, cat="Integer"
)
cal_crise = crise.add_variable(
    "Calcas_Crise", lowBound=0, cat="Integer"
)

# Maximizar lucro
crise += 34.90 * cam_crise + 79.90 * cal_crise

# Restrições
crise += 55 * cam_crise + 100 * cal_crise <= 2500
crise += 15 * cam_crise + 35 * cal_crise <= 300
crise += cam_crise <= 30
crise += cal_crise <= 15

resultado_crise = crise.solve(pulp.COIN_CMD(msg=False))

print("Status:", resultado_crise.status_str)

if resultado_crise.has_solution:
    print("Camisetas:", int(cam_crise.varValue))
    print("Calças:", int(cal_crise.varValue))
    print(f"Lucro na crise: R$ {resultado_crise.objective:.2f}")


Status: Optimal
Camisetas: 20
Calças: 0
Lucro na crise: R$ 698.00


### 4.1 — Análise de Maximização de Lucro

Utilizei a biblioteca PuLP no Python para determinar a quantidade ideal de camisetas oversized e calças cargo da Tauste Roupas, considerando restrições de orçamento, tempo e demanda.

No cenário normal, o algoritmo indicou 25 camisetas e 3 calças, proporcionando lucro bruto máximo de R$ 1.112,20.

Na análise What-If, reduzi o tempo disponível de 480 para 300 minutos. O resultado passou para 20 camisetas e nenhuma calça, com lucro de R$ 698,00, uma redução de 37,2%.

Concluí que o tempo de preparação é um fator importante para o lucro da loja. Essa simulação permite que o gerente planeje suas compras e adapte as operações diante da falta de funcionários.


In [5]:

# PROBLEMA 2 — MINIMIZAÇÃO DE CUSTOS

depositos = {"A": 60, "B": 70}
demandas = {"P1": 50, "P2": 35, "P3": 25}

# Custo de transporte por peça
custos = {
    ("A", "P1"): 3, ("A", "P2"): 5, ("A", "P3"): 6,
    ("B", "P1"): 6, ("B", "P2"): 4, ("B", "P3"): 3
}

modelo2 = pulp.LpProblem("Minimizar_Custos", pulp.LpMinimize)

# Quantidade enviada por cada rota
rotas = {
    (i, j): modelo2.add_variable(
        f"Envio_{i}_{j}", lowBound=0, cat="Integer"
    )
    for i in depositos for j in demandas
}

# Função objetivo
modelo2 += pulp.lpSum(
    custos[i, j] * rotas[i, j]
    for i in depositos for j in demandas
)

# Limite de cada depósito
for i in depositos:
    modelo2 += pulp.lpSum(
        rotas[i, j] for j in demandas
    ) <= depositos[i]

# Atender exatamente à demanda
for j in demandas:
    modelo2 += pulp.lpSum(
        rotas[i, j] for i in depositos
    ) == demandas[j]

# Resolver
resultado2 = modelo2.solve(pulp.COIN_CMD(msg=False))

print("Status:", resultado2.status_str)

if resultado2.has_solution:
    for (i, j), quantidade in rotas.items():
        print(f"Depósito {i} -> {j}: {int(quantidade.varValue)} peças")
    print(f"Custo mínimo: R$ {resultado2.objective:.2f}")


Status: Optimal
Depósito A -> P1: 50 peças
Depósito A -> P2: 0 peças
Depósito A -> P3: 0 peças
Depósito B -> P1: 0 peças
Depósito B -> P2: 35 peças
Depósito B -> P3: 25 peças
Custo mínimo: R$ 365.00


In [6]:

# WHAT-IF — REDUÇÃO DA CAPACIDADE DO DEPÓSITO A

depositos_crise = {"A": 40, "B": 70}

crise2 = pulp.LpProblem("Transporte_Crise", pulp.LpMinimize)

rotas_crise = {
    (i, j): crise2.add_variable(
        f"Crise_{i}_{j}", lowBound=0, cat="Integer"
    )
    for i in depositos_crise for j in demandas
}

# Minimizar custos
crise2 += pulp.lpSum(
    custos[i, j] * rotas_crise[i, j]
    for i in depositos_crise for j in demandas
)

# Capacidade dos depósitos
for i in depositos_crise:
    crise2 += pulp.lpSum(
        rotas_crise[i, j] for j in demandas
    ) <= depositos_crise[i]

# Atender à demanda
for j in demandas:
    crise2 += pulp.lpSum(
        rotas_crise[i, j] for i in depositos_crise
    ) == demandas[j]

# Resolver
resultado_crise2 = crise2.solve(pulp.COIN_CMD(msg=False))

print("Status:", resultado_crise2.status_str)

if resultado_crise2.has_solution:
    for (i, j), quantidade in rotas_crise.items():
        print(f"Depósito {i} -> {j}: {int(quantidade.varValue)} peças")

    print(f"Custo na crise: R$ {resultado_crise2.objective:.2f}")
    print(f"Aumento do custo: R$ {resultado_crise2.objective - 365:.2f}")


Status: Optimal
Depósito A -> P1: 40 peças
Depósito A -> P2: 0 peças
Depósito A -> P3: 0 peças
Depósito B -> P1: 10 peças
Depósito B -> P2: 35 peças
Depósito B -> P3: 25 peças
Custo na crise: R$ 395.00
Aumento do custo: R$ 30.00


### 4.2 — Análise de Minimização de Custos

Utilizei a biblioteca PuLP no Python para calcular a distribuição de mercadorias entre dois depósitos e três pontos de venda da Tauste Roupas.

No cenário normal, o algoritmo encontrou um custo mínimo de R$ 365,00, atendendo toda a demanda de 110 peças.

Na análise What-If, reduzi a capacidade do Depósito A de 60 para 40 peças. O custo aumentou para R$ 395,00, uma diferença de 8,2%.

Concluí que a redução da capacidade de transporte exige uma redistribuição das mercadorias, aumentando os custos. Essa simulação ajuda o gerente a planejar alternativas diante de imprevistos.

Todos os valores utilizados são simulados para fins acadêmicos.

### 4.3 — Análise Crítica de Pesquisa Operacional

Os resultados obtidos no Python demonstraram como a otimização pode auxiliar a Tauste Roupas. Na maximização, o lucro bruto ideal foi de R$ 1.112,20, considerando orçamento, tempo e demanda. Na minimização, o menor custo de transporte foi de R$ 365,00, atendendo aos três pontos de venda. Os resultados fazem sentido dentro das condições simuladas, embora os valores reais dependam dos custos e da demanda da empresa.

As análises What-If demonstraram a importância do planejamento. Com menos tempo de preparação, o lucro caiu para R$ 698,00. Com a redução da capacidade de um depósito, o custo de transporte aumentou para R$ 395,00. Concluí que essas simulações ajudam o gerente a antecipar dificuldades, reorganizar recursos e tomar decisões mais seguras diante de imprevistos.